In [2]:
import pandas as pd

df = pd.read_csv("../data/flight_price.csv")

In [5]:
X = df.drop("price", axis=1)
y = df["price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX:")
print(X.head())

print("\ny:")
print(y.head())

X shape: (300153, 11)
y shape: (300153,)

X:
   Unnamed: 0   airline   flight source_city departure_time stops  \
0           0  SpiceJet  SG-8709       Delhi        Evening  zero   
1           1  SpiceJet  SG-8157       Delhi  Early_Morning  zero   
2           2   AirAsia   I5-764       Delhi  Early_Morning  zero   
3           3   Vistara   UK-995       Delhi        Morning  zero   
4           4   Vistara   UK-963       Delhi        Morning  zero   

    arrival_time destination_city    class  duration  days_left  
0          Night           Mumbai  Economy      2.17          1  
1        Morning           Mumbai  Economy      2.33          1  
2  Early_Morning           Mumbai  Economy      2.17          1  
3      Afternoon           Mumbai  Economy      2.25          1  
4        Morning           Mumbai  Economy      2.33          1  

y:
0    5953
1    5953
2    5956
3    5955
4    5955
Name: price, dtype: int64


In [ ]:
X = df.drop(columns=["price", "Unnamed: 0"])
y = df["price"]


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)



In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (240122, 11)
X_test: (60031, 11)
y_train: (240122,)
y_test: (60031,)


In [14]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(X_train[["class"]])

print(encoded[:5])
print(encoder.get_feature_names_out(["class"]))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 2)>
  Coords	Values
  (0, 1)	1.0
  (1, 1)	1.0
  (2, 0)	1.0
  (3, 1)	1.0
  (4, 0)	1.0
['class_Business' 'class_Economy']


In [18]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# numerical columns
numeric_features = ["duration", "days_left"]

# categorical columns
categorical_features = [
    "airline",
    "flight",
    "source_city",
    "departure_time",
    "stops",
    "arrival_time",
    "destination_city",
    "class"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categ", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)

X_train: (240122, 1598)
X_test: (60031, 1598)


In [20]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("preprocessor" , preprocessor),
    ("regressor" , LinearRegression())
   ])

model.fit(X_train , y_train)

y_pred = model.predict(X_test)

print(y_pred[:5])

[ 5283.21817311 52572.95646022 11267.54968796 49371.05265804
  8906.76379315]


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor",RandomForestRegressor(
        n_estimators = 100,
        random_state = 42,
        n_jobs = -1
    ))
])

rf_model.fit(X_train,y_train)

y_pred_rf = rf_model.predict(X_test)

print(y_pred_rf[:5])